In [7]:
import csv
from pathlib import Path
from orders_utils import normalize_city, parse_amount
import json


DATA_DIR = Path.cwd().parent / 'data'
REQUIRED = {'order_id', 'city', 'amount'}


def audit_orders(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    """Берет список словарей. Проверяет наличие проблем через check_row. Если проблемы есть, добавляет их в список словарей
    problems. Если проблем нет, кладет очищенные строки в clean_rows. Возвращает кортеж из 2х списков словарей."""
    problems = []
    clean_rows = []    
    for row in rows:
        check = check_row(row)
        if check:
            problems.extend(check)                
        else:
            clean_row = {'order_id': row['order_id'], 
                         'city': normalize_city(row['city']), 
                         'amount': parse_amount(row['amount'])}
            clean_rows.append(clean_row)     
    return clean_rows, problems


def check_row(row: dict) -> list[dict]:
    """Проверяет в словаре по ключу 'city'- если None или значение '', добавляет 
    словарь с id, названием поля текущим значением и причиной.
     Проверяет значение amount. Если None, добавляет словарь с id, полем, текущим значением и причиной
     Принимает словарь, возвращает список словарей(может быть 0,1,2 проблем)"""
    problems = []
    if row['city'] is None or row['city'].strip() =='':
        city_problem = {'order_id': row['order_id'], 'field': 'city', 'raw_value': row['city'], 'reason': 'пусто'}
        problems.append(city_problem)
    if parse_amount(row['amount']) is None:
        amount_problem = {'order_id': row['order_id'], 'field': 'amount','raw_value': row['amount'], 'reason': 'не число'}
        problems.append(amount_problem)    
    return problems


def read_orders(path: Path) -> list[dict]:
    """Читает CSV с заказами. Падает с ValueError, если нет обязательных столбцов."""
    with open(path, encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        missing = REQUIRED - set(reader.fieldnames)   # fieldnames — список названий столбцов
        if missing:
            raise ValueError(f'в файле {path.name} нет столбцов: {missing}')
        return list(reader)


def write_clean(clean_rows: list[dict], path: Path) -> None:    
    """записывает чистые строки через DictWriter: столбцы order_id, city, amount, заголовок, все строки."""
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['order_id', 'city', 'amount'])
        writer.writeheader()
        writer.writerows(clean_rows)
    print("Файл записан.")

    
def run_audit(raw_path: Path, clean_path: Path, report_path: Path) -> None:
    """Читает сырой файл, проводит аудит, пишет чистый CSV и отчёт JSON."""
    try:
        rows = read_orders(raw_path)
    except ValueError as e:
        print(f'плохой файл: {e}')
    except FileNotFoundError as e:
        print(f'файл не найден: {e.filename}')
    else:
        clean, probs = audit_orders(rows)
        write_clean(clean, clean_path)        
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(probs, f, ensure_ascii=False, indent=2)
        print(f'чистых строк: {len(clean)}, проблем: {len(probs)}')

    


    


In [8]:
def audit_orders_v2(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    """Делит строки на чистые и проблемы."""
    clean_rows = []
    problems = []
    for row in rows:
        check = check_row(row)
        if check:
            problems.extend(check)
        else:            
            clean_rows.append({'order_id': row['order_id'],
                               'city': normalize_city(row['city']),
                               'amount': parse_amount(row['amount'])})
    return clean_rows, problems

rows = read_orders(DATA_DIR / 'orders_raw.csv')
clean, probs = audit_orders_v2(rows)
print(len(clean), len(probs))

report_path = DATA_DIR / 'reports' / 'problems_v2.json'
report_path.parent.mkdir(exist_ok=True)
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(probs, f, ensure_ascii=False, indent=2)

3 2


In [ ]:
def summarize_problems(problems: list[dict], key: str = 'reason') -> dict[str, int]:
    """считает, сколько проблем каждого вида. Возвращает словарь {Проблема: количество}
    Вид — это значение по ключу key: по умолчанию 'reason', но можно передать 'field'"""        
    counts = {}
    for problem in problems:
        counts[problem[key]] = counts.get(problem[key], 0) + 1     
    return counts


test_problems = [
    {'order_id': '201', 'field': 'city',   'raw_value': '',    'reason': 'пусто'},
    {'order_id': '202', 'field': 'amount', 'raw_value': 'abc', 'reason': 'не число'},
    {'order_id': '203', 'field': 'amount', 'raw_value': '-',   'reason': 'не число'},
    {'order_id': '203', 'field': 'city',   'raw_value': None,  'reason': 'пусто'},
    {'order_id': '204', 'field': 'amount', 'raw_value': '1,5', 'reason': 'не число'},
    {'order_id': '205', 'field': 'city',   'raw_value': '  ',  'reason': 'пусто'},
    {'order_id': '206', 'field': 'amount', 'raw_value': '',    'reason': 'пусто'},
]

print(summarize_problems(test_problems))           # {'пусто': 4, 'не число': 3}
print(summarize_problems(test_problems, 'field'))  # {'city': 3, 'amount': 4}
print(summarize_problems(probs))                   # {'пусто': 1, 'не число': 1}
print(summarize_problems([]))                      # {}

print(summarize_problems(probs + [{'order_id': '101', 'field': 'order_id',
                                   'raw_value': '101', 'reason': 'дубль'}]))
# {'пусто': 1, 'не число': 1, 'дубль': 1}
    

{'пусто': 4, 'не число': 3}
{'city': 3, 'amount': 4}
{'пусто': 1, 'не число': 1}
{}
{'пусто': 1, 'не число': 1, 'дубль': 1}


In [27]:
test_problems = [
    {'order_id': '201', 'field': 'city',   'raw_value': '',    'reason': 'пусто'},
    {'order_id': '202', 'field': 'amount', 'raw_value': 'abc', 'reason': 'не число'},
    {'order_id': '203', 'field': 'amount', 'raw_value': '-',   'reason': 'не число'},
    {'order_id': '203', 'field': 'city',   'raw_value': None,  'reason': 'пусто'},
    {'order_id': '204', 'field': 'amount', 'raw_value': '1,5', 'reason': 'не число'},
    {'order_id': '205', 'field': 'city',   'raw_value': '  ',  'reason': 'пусто'},
    {'order_id': '206', 'field': 'amount', 'raw_value': '',    'reason': 'пусто'},
]

print(summarize_problems(test_problems))           # {'пусто': 4, 'не число': 3}
print(summarize_problems(test_problems, 'field'))  # {'city': 3, 'amount': 4}
print(summarize_problems(probs))                   # {'пусто': 1, 'не число': 1}
print(summarize_problems([]))                      # {}


{'пусто': 4, 'не число': 3}
{'city': 3, 'amount': 4}
{'пусто': 1, 'не число': 1}
{}


In [ ]:
o ={'order_id': '201', 'field': 'city',   'raw_value': '',    'reason': 'пусто'}
total_problems = {}

total_problems[o['reason']] =counts.get(item, 0) + 1
print(total_problems)

{'пусто': 'кря'}
